<a href="https://colab.research.google.com/github/edirnolove/FDG-2026/blob/main/semana04/EDIR_LAB04_FGD_INDIVIDUAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LABORATORIO DIRIGIDO N.° 04
**Semana 4 · Modelado Relacional**  
Panadería Dulce Migaja: del registro de ventas a tablas relacionadas en SQLite  
Curso: Fundamentos de Gestión de Datos · Periodo: 2026-II · Docente: PILAR ROCIO SAYAN MEJIA  
Ejecute las celdas una vez y en orden, desde una sesión nueva.

## Actividad 1 — Revisión de conceptos

**Instrucciones:** complete la tabla con definiciones elaboradas con sus propias palabras. No se admite la reproducción literal de fuentes externas ni de sistemas generativos. Si su definición permite responder la pregunta de apoyo, es suficiente.

| Concepto | Definición con sus propias palabras | Pregunta de apoyo |
|---|---|---|
| Entidad y atributo | Una entidad es algo de lo que guardamos datos (cliente, venta) y un atributo es un dato que la describe (nombre, fecha). La venta es una entidad y el distrito del cliente es un atributo. | En Dulce Migaja, ¿la venta es una entidad o un atributo? ¿Y el distrito del cliente? |
| Clave primaria y clave foránea | La PK identifica cada fila y no se repite. La FK apunta a la PK de otra tabla. id_cliente es PK en clientes y FK en ventas, donde se repite porque un cliente compra varias veces. | ¿Por qué id_cliente puede repetirse en ventas y no en clientes? |
| Cardinalidad 1:N y N:M | En 1:N una fila se relaciona con muchas (un cliente, muchas ventas) y la FK va en el lado N, en ventas. En N:M muchas con muchas, y se necesita una tabla intermedia. | ¿Dónde va la FK entre cliente y ventas? ¿Qué se necesita si un producto tiene varias categorías? |
| Dependencia funcional y formas normales | Un dato depende de otro cuando este lo determina. 1FN: un solo valor por celda. 2FN: nada depende de solo una parte de la clave. 3FN: nada depende de un dato que no es la clave. | ¿Qué revisa cada forma normal: 1FN, 2FN y 3FN? |

## Actividad 2 — Del problema al modelo

### Caso de estudio: Panadería Dulce Migaja

Panadería Dulce Migaja tiene tres locales en Lima Norte y registraba sus ventas en una sola tabla, ventas_original. Cada fila es la venta de un solo producto a un cliente en un local. La tabla muestra las primeras 6 de sus 18 ventas.

**Naturaleza de los datos.** Caso simulado (no son ventas ni personas reales), con 18 ventas del 1 al 6 de abril de 2026. La base del laboratorio proviene de: https://raw.githubusercontent.com/Rociosayan/PMD1_Fundamentos_Gestion_Datos/main/casos/17_panaderia_normalizacion/panaderia_normalizacion.db. No se descarga: en la Actividad 4 las tablas se construyen en memoria con los mismos datos.

| Venta | Fecha | Cliente / distrito | Producto / categoría | Local / distrito | Cant. | Precio S/ |
|---|---|---|---|---|---|---|
| 1 | 2026-04-01 | Ana Torres / Los Olivos | Pan frances / Panes | Panaderia Central / Los Olivos | 20 | 0.5 |
| 2 | 2026-04-01 | Luis Ramos / Comas | Empanada de pollo / Salados | Panaderia Norte / Comas | 4 | 4.5 |
| 3 | 2026-04-01 | Maria Perez / Independencia | Jugo de naranja / Bebidas | Panaderia Plaza / Independencia | 2 | 5.0 |
| 4 | 2026-04-02 | Ana Torres / Los Olivos | Torta de chocolate / Pasteles | Panaderia Central / Los Olivos | 1 | 8.0 |
| 5 | 2026-04-02 | Carlos Rios / San Martin de Porres | Pan integral / Panes | Panaderia Norte / Comas | 12 | 0.8 |
| 6 | 2026-04-02 | Rosa Diaz / Los Olivos | Cafe americano / Bebidas | Panaderia Central / Los Olivos | 3 | 4.0 |

**Pregunta 1.** ¿Qué representa una fila de ventas_original y qué atributos pertenecen realmente a la venta?

**Respuesta:** Una fila es la venta de un producto a un cliente en un local. Son atributos de la venta: número de venta, fecha, cantidad y método de pago.

**Pregunta 2.** ¿Qué datos descriptivos se repiten y qué ocurre si Ana Torres cambia de distrito?

**Respuesta:** Se repiten los datos del cliente, del producto con su categoría y del local. Ana Torres aparece en 2 de las 6 filas mostradas (ventas 1 y 4); si cambia de distrito hay que corregir todas sus filas, y si se olvida una queda información contradictoria.

### Modelo de referencia

Separe las entidades e indique sus atributos, PK, FK y relaciones. Cada venta referencia un cliente, un producto y un local; un cliente, producto o local puede existir sin ventas.

| Entidad | Atributos | PK | FK | Relación |
|---|---|---|---|---|
| categorias | id_categoria, nombre | id_categoria | Ninguna | 1 categoría : N productos |
| productos | id_producto, nombre, id_categoria, precio_unitario | id_producto | id_categoria | N productos : 1 categoría |
| clientes | id_cliente, nombre, distrito | id_cliente | Ninguna | 1 cliente : N ventas |
| locales | id_local, nombre, distrito | id_local | Ninguna | 1 local : N ventas |
| ventas | id_venta, fecha, id_cliente, id_producto, id_local, cantidad, metodo_pago | id_venta | id_cliente, id_producto, id_local | N ventas : 1 cliente, 1 producto y 1 local |

**Se solicita.** Elabore el diagrama ER en https://app.diagrams.net/ con PK, FK y cardinalidades, e inserte la imagen o el enlace en el notebook. Link del diagrama: https://drive.google.com/file/d/1nOt1NViwmWaqJ55l1QSaY_8eIxyYmS_H/view?usp=sharing

**Pregunta 3.** Ejercicio 1:1. Formule una regla entre empleado y credencial vigente que permita como máximo una credencial vigente por empleado. ¿Qué restricción la implementa?

**Respuesta:** Regla: un empleado tiene como máximo una credencial vigente. Se implementa con UNIQUE sobre id_empleado en la tabla credenciales.

**Pregunta 4.** Ejercicio N:M. Si un producto pudiera pertenecer a varias categorías, ¿qué tabla propone y cuál sería su clave?

**Respuesta:** Una tabla intermedia, producto_categoria, con clave compuesta (id_producto, id_categoria).

## Actividad 3 — Ejercicios de normalización

**Indicaciones.** Son variantes conceptuales: no modifican el modelo principal. Que un texto se repita no basta para afirmar que una tabla incumple una forma normal; revise valores atómicos, la clave y las dependencias.

### Ejercicio 1FN

Muestra didáctica simulada para esta explicación: un pedido de Ana Torres registra en una sola celda «Pan Integral, Café».

**Pregunta 5.** Convierta la celda en filas atómicas. ¿Cuál es la clave de las filas resultantes y por qué id_pedido solo no basta?

**Respuesta:** La celda «Pan Integral, Café» se separa en dos filas: una con Ana Torres y Pan Integral, y otra con Ana Torres y Café, ambas con el mismo id_pedido. La clave es compuesta: (id_pedido, producto). id_pedido solo no basta porque se repite en las dos filas, ya que un pedido tiene varios productos.

### Ejercicio 2FN

Variante conceptual: detalle_venta(id_venta, id_producto, nombre_producto, cantidad), con clave (id_venta, id_producto); cada producto aparece una sola vez por venta.

**Pregunta 6.** Identifique la dependencia parcial y proponga las tablas resultantes.

**Respuesta:** La dependencia parcial es id_producto → nombre_producto: el nombre depende solo de una parte de la clave (id_venta, id_producto). Se separa en dos tablas: productos(id_producto, nombre_producto), con PK id_producto, y detalle_venta(id_venta, id_producto, cantidad), con PK (id_venta, id_producto) y FK id_producto.

### Ejercicio 3FN

productos(id_producto, nombre, id_categoria, nombre_categoria, precio_unitario), con las reglas id_producto → id_categoria e id_categoria → nombre_categoria.

**Pregunta 7.** Explique la dependencia transitiva y proponga la separación.

**Respuesta:** Hay dependencia transitiva porque id_producto → id_categoria e id_categoria → nombre_categoria. Entonces nombre_categoria depende de id_producto de forma indirecta, a través de id_categoria, que no es la clave. Se separa en categorias(id_categoria, nombre_categoria) y productos(id_producto, nombre, id_categoria, precio_unitario), con FK id_categoria.

## Actividad 4 — Construcción guiada en SQLite

Ejecute las celdas una vez y en orden, desde una sesión nueva. La base en memoria existe mientras la conexión esté abierta: si reinicia el entorno, vuelva a ejecutar desde la celda 1. Cada resultado se muestra con pd.read_sql_query.

### A. Crear la base y las tablas

### Paso 1 — Conexión y verificación de claves foráneas

sqlite3.connect(":memory:") crea una base vacía en memoria. PRAGMA foreign_keys = ON activa la comprobación de claves foráneas, que en SQLite viene desactivada.

**Celda 1 de 15**

In [ ]:
import sqlite3
import pandas as pd

# Base en memoria: existe mientras la conexión esté abierta
conexion = sqlite3.connect(":memory:")
conexion.execute("PRAGMA foreign_keys = ON;")
pd.read_sql_query("PRAGMA foreign_keys;", conexion)

,foreign_keys
0,1


### Paso 2 — Tabla sin dependencias: categorias

CREATE TABLE define columnas y restricciones: INTEGER PRIMARY KEY identifica cada fila, NOT NULL exige un valor y UNIQUE impide nombres repetidos. INSERT INTO agrega filas y commit confirma los cambios.

**Celda 2 de 15**

In [ ]:
conexion.execute("""CREATE TABLE categorias (
    id_categoria INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL UNIQUE
)""")
conexion.execute("""INSERT INTO categorias (id_categoria, nombre)
VALUES (1, 'Panes'), (2, 'Pasteles'), (3, 'Bebidas'), (4, 'Salados')""")
conexion.commit()
pd.read_sql_query("SELECT * FROM categorias", conexion)

,id_categoria,nombre
0,1,Panes
1,2,Pasteles
2,3,Bebidas
3,4,Salados


### Paso 3 — productos con su clave foránea

FOREIGN KEY indica la columna que apunta a otra tabla y REFERENCES, la tabla y la PK referenciadas. REAL guarda números con decimales.

**Celda 3 de 15**

In [ ]:
conexion.execute("""CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    id_categoria INTEGER NOT NULL,
    precio_unitario REAL NOT NULL,
    FOREIGN KEY (id_categoria)
        REFERENCES categorias(id_categoria)
)""")
conexion.execute("""INSERT INTO productos
    (id_producto, nombre, id_categoria, precio_unitario)
VALUES
    (1, 'Pan frances', 1, 0.5),
    (2, 'Pan integral', 1, 0.8),
    (3, 'Empanada de pollo', 4, 4.5),
    (4, 'Torta de chocolate', 2, 8.0),
    (5, 'Jugo de naranja', 3, 5.0),
    (6, 'Cafe americano', 3, 4.0)""")
conexion.commit()
pd.read_sql_query("SELECT * FROM productos", conexion)

,id_producto,nombre,id_categoria,precio_unitario
0,1,Pan frances,1,0.5
1,2,Pan integral,1,0.8
2,3,Empanada de pollo,4,4.5
3,4,Torta de chocolate,2,8.0
4,5,Jugo de naranja,3,5.0
5,6,Cafe americano,3,4.0


**Pregunta 8.** ¿Por qué categorias se crea antes que productos? ¿Qué productos pertenecen a la categoría 3?

**Respuesta:** categorias se crea antes porque productos tiene una FK que apunta a categorias(id_categoria), y la tabla referenciada debe existir primero. A la categoría 3 pertenecen Jugo de naranja (id 5) y Cafe americano (id 6).

### Paso 4 — clientes y locales

Dos tablas sin FK. Dos personas pueden compartir nombre y distrito; por eso la identidad del cliente es id_cliente y no su nombre.

**Celda 4 de 15**

In [ ]:
conexion.execute("""CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    distrito TEXT NOT NULL
)""")
conexion.execute("""CREATE TABLE locales (
    id_local INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    distrito TEXT NOT NULL
)""")
conexion.execute("""INSERT INTO clientes (id_cliente, nombre, distrito)
VALUES
    (1, 'Ana Torres', 'Los Olivos'),
    (2, 'Luis Ramos', 'Comas'),
    (3, 'Maria Perez', 'Independencia'),
    (4, 'Carlos Rios', 'San Martin de Porres'),
    (5, 'Rosa Diaz', 'Los Olivos'),
    (6, 'Jorge Salas', 'Comas')""")
conexion.execute("""INSERT INTO locales (id_local, nombre, distrito)
VALUES
    (1, 'Panaderia Central', 'Los Olivos'),
    (2, 'Panaderia Norte', 'Comas'),
    (3, 'Panaderia Plaza', 'Independencia')""")
conexion.commit()
pd.read_sql_query("SELECT * FROM clientes", conexion)

,id_cliente,nombre,distrito
0,1,Ana Torres,Los Olivos
1,2,Luis Ramos,Comas
2,3,Maria Perez,Independencia
3,4,Carlos Rios,San Martin de Porres
4,5,Rosa Diaz,Los Olivos
5,6,Jorge Salas,Comas


**Pregunta 9.** ¿En qué tabla y en cuántas filas se corrige ahora el distrito de Ana Torres?

**Respuesta:** Se corrige en la tabla clientes y en 1 sola fila (id_cliente 1), porque el distrito se guarda una sola vez.

### Paso 5 — ventas con tres claves foráneas

Una fila es la venta de un producto a un cliente en un local. Las tres FK conectan entidades; fecha (TEXT, AAAA-MM-DD), cantidad y metodo_pago son atributos propios. No se guarda un total: se calcula. PRAGMA table_info muestra la estructura creada.

**Celda 5 de 15**

In [ ]:
conexion.execute("""CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY,
    fecha TEXT NOT NULL,
    id_cliente INTEGER NOT NULL,
    id_producto INTEGER NOT NULL,
    id_local INTEGER NOT NULL,
    cantidad INTEGER NOT NULL,
    metodo_pago TEXT NOT NULL,
    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente),
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto),
    FOREIGN KEY (id_local) REFERENCES locales(id_local)
)""")
conexion.commit()
pd.read_sql_query("PRAGMA table_info(ventas);", conexion)

,cid,name,type,notnull,dflt_value,pk
0,0,id_venta,INTEGER,0,None,1
1,1,fecha,TEXT,1,None,0
2,2,id_cliente,INTEGER,1,None,0
3,3,id_producto,INTEGER,1,None,0
4,4,id_local,INTEGER,1,None,0
5,5,cantidad,INTEGER,1,None,0
6,6,metodo_pago,TEXT,1,None,0


### B. Insertar y comprobar

### Paso 6 — Insertar las 18 ventas

La lista de columnas y cada fila de valores deben tener el mismo orden. Un mismo id_cliente puede aparecer en varias ventas. La consulta final muestra las ventas del cliente 1.

**Celda 6 de 15**

In [ ]:
conexion.execute("""INSERT INTO ventas
    (id_venta, fecha, id_cliente, id_producto,
     id_local, cantidad, metodo_pago)
VALUES
    (1, '2026-04-01', 1, 1, 1, 20, 'Efectivo'),
    (2, '2026-04-01', 2, 3, 2, 4, 'Yape'),
    (3, '2026-04-01', 3, 5, 3, 2, 'Tarjeta'),
    (4, '2026-04-02', 1, 4, 1, 1, 'Tarjeta'),
    (5, '2026-04-02', 4, 2, 2, 12, 'Efectivo'),
    (6, '2026-04-02', 5, 6, 1, 3, 'Yape'),
    (7, '2026-04-03', 2, 1, 2, 15, 'Efectivo'),
    (8, '2026-04-03', 6, 3, 3, 2, 'Yape'),
    (9, '2026-04-03', 3, 4, 3, 2, 'Tarjeta'),
    (10, '2026-04-04', 4, 5, 2, 4, 'Yape'),
    (11, '2026-04-04', 5, 1, 1, 25, 'Efectivo'),
    (12, '2026-04-04', 6, 2, 3, 10, 'Tarjeta'),
    (13, '2026-04-05', 1, 3, 1, 3, 'Efectivo'),
    (14, '2026-04-05', 2, 6, 2, 2, 'Yape'),
    (15, '2026-04-05', 3, 1, 3, 18, 'Efectivo'),
    (16, '2026-04-06', 4, 4, 2, 1, 'Tarjeta'),
    (17, '2026-04-06', 5, 5, 1, 2, 'Yape'),
    (18, '2026-04-06', 6, 3, 3, 4, 'Efectivo')""")
conexion.commit()
pd.read_sql_query("SELECT * FROM ventas WHERE id_cliente = 1", conexion)

,id_venta,fecha,id_cliente,id_producto,id_local,cantidad,metodo_pago
0,1,2026-04-01,1,1,1,20,Efectivo
1,4,2026-04-02,1,4,1,1,Tarjeta
2,13,2026-04-05,1,3,1,3,Efectivo


**Pregunta 10.** En la celda 5, ¿qué columna tiene pk = 1 y cuáles notnull = 1? Según la celda 6, ¿cuántas ventas tiene Ana Torres y por qué repetir su id_cliente no es un error?

**Respuesta:** En la celda 5 tiene pk = 1 la columna id_venta. Tienen notnull = 1 fecha, id_cliente, id_producto, id_local, cantidad y metodo_pago. Según la celda 6, Ana Torres tiene 3 ventas (1, 4 y 13). Repetir su id_cliente no es un error porque en ventas es una FK: un cliente puede comprar muchas veces.

### Paso 7 — Conteos e integridad referencial

COUNT(*) cuenta filas y cada subconsulta devuelve un número. PRAGMA foreign_key_check busca FK que apunten a registros inexistentes: si no devuelve filas, no hay incidencias.

**Celda 7 de 15**

In [ ]:
consulta = """SELECT
    (SELECT COUNT(*) FROM categorias) AS categorias,
    (SELECT COUNT(*) FROM productos) AS productos,
    (SELECT COUNT(*) FROM clientes) AS clientes,
    (SELECT COUNT(*) FROM locales) AS locales,
    (SELECT COUNT(*) FROM ventas) AS ventas"""
pd.read_sql_query(consulta, conexion)

,categorias,productos,clientes,locales,ventas
0,4,6,6,3,18


**Celda 8 de 15**

In [ ]:
# Sin filas = ninguna FK apunta a un registro inexistente
pd.read_sql_query("PRAGMA foreign_key_check;", conexion)

,table,rowid,parent,fkid


**Pregunta 11.** Anote los conteos. ¿Qué devolvió la celda 8? ¿Por qué contar filas no basta para validar el diseño?

**Respuesta:** Los conteos son: categorias 4, productos 6, clientes 6, locales 3 y ventas 18. La celda 8 devolvió una tabla sin filas, es decir, ninguna FK apunta a un registro que no existe. Contar filas no basta porque solo dice cuántos registros hay, no si las relaciones son correctas; eso lo comprueba foreign_key_check.

### C. Comprobar el diseño con JOIN

### Paso 8 — Reconstruir el reporte de ventas

Cada ON une una FK con su PK. Alias: v = ventas, c = clientes, p = productos, cat = categorias y l = locales. El importe asume precios constantes: si el precio cambia, un sistema que registra ventas históricas debe conservar el precio aplicado en cada venta.

**Celda 9 de 15**

In [ ]:
consulta = """SELECT v.id_venta, v.fecha,
       c.nombre AS cliente,
       p.nombre AS producto,
       cat.nombre AS categoria,
       l.nombre AS local,
       v.cantidad,
       ROUND(v.cantidad * p.precio_unitario, 2) AS importe
FROM ventas AS v
INNER JOIN clientes AS c ON v.id_cliente = c.id_cliente
INNER JOIN productos AS p ON v.id_producto = p.id_producto
INNER JOIN categorias AS cat ON p.id_categoria = cat.id_categoria
INNER JOIN locales AS l ON v.id_local = l.id_local
ORDER BY v.id_venta"""
pd.read_sql_query(consulta, conexion)

,id_venta,fecha,cliente,producto,categoria,local,cantidad,importe
0,1,2026-04-01,Ana Torres,Pan frances,Panes,Panaderia Central,20,10.0
1,2,2026-04-01,Luis Ramos,Empanada de pollo,Salados,Panaderia Norte,4,18.0
2,3,2026-04-01,Maria Perez,Jugo de naranja,Bebidas,Panaderia Plaza,2,10.0
3,4,2026-04-02,Ana Torres,Torta de chocolate,Pasteles,Panaderia Central,1,8.0
4,5,2026-04-02,Carlos Rios,Pan integral,Panes,Panaderia Norte,12,9.6
5,6,2026-04-02,Rosa Diaz,Cafe americano,Bebidas,Panaderia Central,3,12.0
6,7,2026-04-03,Luis Ramos,Pan frances,Panes,Panaderia Norte,15,7.5
7,8,2026-04-03,Jorge Salas,Empanada de pollo,Salados,Panaderia Plaza,2,9.0
8,9,2026-04-03,Maria Perez,Torta de chocolate,Pasteles,Panaderia Plaza,2,16.0
9,10,2026-04-04,Carlos Rios,Jugo de naranja,Bebidas,Panaderia Norte,4,20.0


**Pregunta 12.** ¿Por qué se usan cuatro JOIN? Compare el importe de la venta 2 con la tabla ventas_original.

**Respuesta:** Se usan cuatro JOIN porque ventas solo guarda ids: se une con clientes, productos y locales, y la categoría se obtiene uniendo productos con categorias. El importe de la venta 2 es 18.0 (4 × 4.5) y coincide con el de ventas_original.

### Paso 9 — Registros, unidades e ingreso

COUNT cuenta registros; SUM(cantidad) suma unidades; el ingreso multiplica la cantidad por el precio unitario.

**Celda 10 de 15**

In [ ]:
consulta = """SELECT p.nombre AS producto,
       COUNT(v.id_venta) AS registros_venta,
       SUM(v.cantidad) AS unidades,
       ROUND(SUM(v.cantidad * p.precio_unitario), 2) AS ingreso
FROM ventas AS v
INNER JOIN productos AS p ON v.id_producto = p.id_producto
GROUP BY p.id_producto, p.nombre
ORDER BY ingreso DESC, p.id_producto"""
pd.read_sql_query(consulta, conexion)

,producto,registros_venta,unidades,ingreso
0,Empanada de pollo,4,13,58.5
1,Jugo de naranja,3,8,40.0
2,Pan frances,4,78,39.0
3,Torta de chocolate,3,4,32.0
4,Cafe americano,2,5,20.0
5,Pan integral,2,22,17.6


**Pregunta 13.** Con Pan frances, explique la diferencia entre registros_venta y unidades. ¿El producto con mayor ingreso es el más rentable?

**Respuesta:** Pan frances tiene 4 registros_venta, que son las veces que aparece en una venta, y 78 unidades, que son los panes vendidos en total. El producto con mayor ingreso es Empanada de pollo (58.5), pero eso no demuestra que sea el más rentable, porque el ingreso no considera los costos.

## Actividad 5 — Retos individuales

**Indicaciones.** Resuelva cada reto de forma individual, en el mismo notebook y después de la celda 10. Use solo conexion.execute, conexion.commit y pd.read_sql_query; ejecute cada reto una sola vez y en orden. Los retos son la evidencia de la actividad individual.

### Reto 1 — Relación N:M: producto en varias categorías

Dulce Migaja quiere que algunos productos aparezcan también en una categoría Promociones. Cree la tabla intermedia producto_categoria, con PK compuesta y dos FK, e inserte cuatro filas de prueba.

**Celda 11 de 15**

In [ ]:
# Reto 1: relación N:M producto–categoría
# 1. Inserte en categorias la categoría 5, 'Promociones'.
# 2. Cree producto_categoria(id_producto, id_categoria) con PK compuesta y dos FK.
# 3. Inserte las filas (4, 2), (4, 5), (5, 3) y (5, 5) y confirme con conexion.commit().
# 4. Muestre producto y categoría uniendo las tres tablas con INNER JOIN.
# Escriba su código aquí y termine con pd.read_sql_query(consulta, conexion)

**Pregunta 14.** ¿Cuál es la PK de producto_categoria y por qué es compuesta? ¿Qué productos quedaron en dos categorías?

**Respuesta:** La PK es (id_producto, id_categoria). Es compuesta porque ninguna columna sola identifica la fila: un producto aparece en varias filas y una categoría también; solo la pareja no se repite. Quedaron en dos categorías la Torta de chocolate (Pasteles y Promociones) y el Jugo de naranja (Bebidas y Promociones).

### Reto 2 — Registrar una venta nueva

El 7 de abril Luis Ramos compra 6 unidades de Pan integral en Panaderia Norte y paga con Yape. Registre la venta 19 con INSERT INTO ... VALUES y compruébela con un JOIN.

**Celda 12 de 15**

In [ ]:
# Reto 2: venta nueva
# 1. Inserte la venta 19: '2026-04-07', cliente 2, producto 2, local 2, cantidad 6, 'Yape'.
#    Use INSERT INTO ventas con la lista de columnas y conexion.commit().
# 2. Compruébela con un JOIN a clientes, productos y locales (WHERE v.id_venta = 19),
#    mostrando el importe con ROUND(v.cantidad * p.precio_unitario, 2).
# Escriba su código aquí y termine con pd.read_sql_query(consulta, conexion)

**Pregunta 15.** ¿Qué importe tiene la venta 19? ¿Qué respondería SQLite si la venta se registrara con id_cliente = 9?

**Respuesta:** La venta 19 tiene un importe de 4.8 (6 unidades × 0.8). Con id_cliente = 9, SQLite la rechazaría con el error "FOREIGN KEY constraint failed", porque no existe el cliente 9.

### Reto 3 — Ventas por local

La gerencia quiere comparar los tres locales: número de ventas, unidades e ingreso de cada uno.

**Celda 13 de 15**

In [ ]:
# Reto 3: por cada local, número de ventas (COUNT), unidades (SUM de cantidad)
# e ingreso (SUM de cantidad * precio_unitario), ordenado de mayor a menor ingreso.
# Escriba su consulta aquí y termine con pd.read_sql_query(consulta, conexion)

**Pregunta 16.** ¿Qué local tiene más ventas e ingreso? ¿Coincide con el que vende más unidades? Explique la diferencia.

**Respuesta:** Panaderia Norte tiene más ventas (7) y más ingreso (75.9). No coincide con el que vende más unidades, que es Panaderia Central (54). Central vende muchas unidades de pan, que es barato, y por eso su ingreso (66.0) es menor.

### Reto 4 — Ingreso por categoría

Compras necesita saber qué categorías generan más ingreso. La categoría de cada venta se obtiene desde productos.

**Celda 14 de 15**

In [ ]:
# Reto 4: unidades e ingreso por categoría, uniendo ventas, productos y categorias.
# Escriba su consulta aquí y termine con pd.read_sql_query(consulta, conexion)

**Pregunta 17.** ¿Qué categoría genera más ingreso y por qué Panes y Bebidas tienen ingresos parecidos con unidades tan distintas?

**Respuesta:** Panes genera más ingreso (61.4). Panes y Bebidas tienen ingresos parecidos (61.4 y 60.0) aunque Panes vendió 106 unidades y Bebidas 13, porque los panes cuestan poco (0.5 y 0.8) y las bebidas cuestan más (5.0 y 4.0).

### Reto 5 — Compras por cliente

Marketing quiere conocer cuántas compras e ingreso aporta cada cliente, con su distrito.

**Celda 15 de 15**

In [ ]:
# Reto 5: por cada cliente, distrito, número de compras e ingreso,
# ordenado de mayor a menor ingreso.
# Escriba su consulta aquí y termine con pd.read_sql_query(consulta, conexion)

**Pregunta 18.** ¿Qué cliente aporta más ingreso? ¿Por qué se agrupa por id_cliente y no solo por nombre?

**Respuesta:** Luis Ramos (id 2) aporta más ingreso, con 38.3 en 4 compras. Se agrupa por id_cliente porque el nombre puede repetirse entre personas distintas; el id identifica a cada cliente y evita juntar clientes diferentes en un mismo grupo.

## Conclusiones

Escriba tres conclusiones técnicas y de negocio de este laboratorio.

1. Separar ventas_original en cinco tablas evita repetir datos: el distrito de un cliente se corrige en 1 sola fila.

2. Las PK y FK mantienen la integridad: SQLite rechaza una venta con un cliente que no existe, y los JOIN reconstruyen el reporte completo.

3. Panaderia Norte lidera en ingreso (75.9) y Panes en unidades (106). Mayor ingreso no demuestra mayor rentabilidad, y los datos son simulados.

## Criterios de evaluación

Rúbrica holística del Laboratorio Dirigido N.° 04 (individual). Puntaje total: 20 puntos.

**Elemento de la capacidad:** diseña un modelo relacional sencillo, aplica 1FN–3FN, implementa tablas SQLite con PK, FK y datos de prueba, y comprueba el diseño con una consulta JOIN.

| Criterio | Excelente | Bueno | Requiere mejora | No aceptable |
|---|---|---|---|---|
| Revisión de conceptos (Actividad 1) · 2 puntos | Define con sus propias palabras entidad, atributo, PK, FK, cardinalidades y formas normales, y responde las preguntas de apoyo. | Define correctamente la mayoría de los conceptos, con imprecisiones menores. | Presenta definiciones parciales, confusas o copiadas. | No completa la actividad o las respuestas no corresponden. |
| Del problema al modelo (Actividad 2) · 3 puntos | Identifica entidades, atributos, PK, FK y cardinalidades, explica las anomalías con cifras y elabora un ER coherente con el modelo. | Identifica la mayoría de los elementos del modelo; el ER presenta omisiones menores. | Confunde entidades y atributos o ubica mal las FK; el ER es incompleto. | No elabora el modelo ni el ER. |
| Normalización (Actividad 3) · 3 puntos | Identifica la clave y las dependencias, y separa correctamente las tablas en 1FN, 2FN y 3FN. | Resuelve la mayoría de los ejercicios con imprecisiones menores en las dependencias. | Separa tablas sin justificar dependencias o confunde parcial y transitiva. | No resuelve los ejercicios. |
| Construcción guiada en SQLite (Actividad 4) · 5 puntos | Ejecuta las celdas en orden, explica PK, FK y restricciones, interpreta conteos, integridad y JOIN, y distingue registros, unidades e ingreso. | Ejecuta las celdas e interpreta la mayoría de los resultados con apoyo parcial. | Ejecuta pocas celdas o interpreta los resultados de forma incorrecta. | No ejecuta las celdas ni interpreta los resultados. |
| Retos individuales y enlace al notebook (Colab o GitHub) (Actividad 5) · 5 puntos | Resuelve los cinco retos con código correcto, interpreta cada resultado con cifras y entrega un enlace funcional al notebook ejecutado. | Resuelve la mayoría de los retos e interpreta los resultados con imprecisiones menores; el enlace funciona. | Resuelve pocos retos o no interpreta los resultados; el enlace está incompleto. | No presenta retos ni un enlace funcional. |
| Conclusiones · 2 puntos | Redacta tres conclusiones técnicas y de negocio, con cifras, ortografía y redacción adecuadas. | Redacta conclusiones adecuadas, con errores menores de redacción. | Las conclusiones son generales o presentan errores frecuentes. | No redacta conclusiones. |

**Total:** ________ / 20

| Nivel | Descripción |
|---|---|
| Excelente | Demuestra un completo entendimiento del problema o realiza la actividad cumpliendo todos los requerimientos especificados. |
| Bueno | Demuestra un considerable entendimiento del problema o realiza la actividad cumpliendo con la mayoría de los requerimientos especificados. |
| Requiere mejora | Demuestra un bajo entendimiento del problema o realiza la actividad con pocos de los requerimientos especificados. |
| No aceptable | No demuestra entendimiento del problema o actividad. |